In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
import plotly.express as px
import plotly.graph_objects as go

In [4]:
co2 = pd.read_csv("CO2.csv")

In [5]:
co2

,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
0,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
2,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
3,ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244
...,...,...,...,...,...,...,...,...,...,...,...,...
7380,VOLVO,XC40 T5 AWD,SUV - SMALL,2.0,4,AS8,Z,10.7,7.7,9.4,30,219
7381,VOLVO,XC60 T5 AWD,SUV - SMALL,2.0,4,AS8,Z,11.2,8.3,9.9,29,232
7382,VOLVO,XC60 T6 AWD,SUV - SMALL,2.0,4,AS8,Z,11.7,8.6,10.3,27,240
7383,VOLVO,XC90 T5 AWD,SUV - STANDARD,2.0,4,AS8,Z,11.2,8.3,9.9,29,232


## 1) Top 5 vehicle makes and models with the highest CO2 emissions and fuel type with the lowest average emissions.

In [10]:
# Top 5 vehicle makes and models with the highest CO2 emissions
top_5_emissions = co2[['Make', 'Model', 'CO2 Emissions(g/km)']].sort_values(
    by='CO2 Emissions(g/km)', ascending=False).head(5)
print("Top 5 vehicles with the highest CO2 emissions:")
print(top_5_emissions)

# Fuel type with the lowest average emissions
lowest_emission_fuel = co2.groupby('Fuel Type')['CO2 Emissions(g/km)'].mean().idxmin()
print(f"Fuel type with the lowest average emissions: {lowest_emission_fuel}")

Top 5 vehicles with the highest CO2 emissions:
             Make               Model  CO2 Emissions(g/km)
5575      BUGATTI              Chiron                  522
6640      BUGATTI              Chiron                  522
4509      BUGATTI              CHIRON                  522
7059  LAMBORGHINI  Aventador Roadster                  493
6046  LAMBORGHINI  Aventador Roadster                  493
Fuel type with the lowest average emissions: N


### Bar chart

In [22]:
fig = px.bar(
    top_5_emissions,
    x='CO2 Emissions(g/km)',
    y='Model',
    orientation='h',
    color='Make',
    title='Top 5 Vehicles with Highest CO2 Emissions',
    labels={'CO2 Emissions(g/km)': 'CO2 Emissions (g/km)', 'Model': 'Vehicle Model'}
)
fig.show()

In [27]:
fuel_avg_emissions = co2.groupby('Fuel Type')['CO2 Emissions(g/km)'].mean().reset_index()
fig = px.pie(
    fuel_avg_emissions,
    values='CO2 Emissions(g/km)',
    names='Fuel Type',
    title='Average CO2 Emissions by Fuel Type',
    color_discrete_sequence=px.colors.sequential.Viridis
)
fig.show()

## 2) CO2 emissions for manual vs. automatic transmissions and Group vehicles by engine size range (e.g., <2.0L, 2.0–3.0L, >3.0L) and analyze their average CO2 emissions.

In [14]:
# Compare CO2 emissions for manual vs. automatic transmissions
co2['Transmission_Type'] = co2['Transmission'].apply(lambda x: 'Manual' if 'M' in x else 'Automatic')
transmission_avg_emissions = co2.groupby('Transmission_Type')['CO2 Emissions(g/km)'].mean()
print("Average CO2 emissions by transmission type:")
print(transmission_avg_emissions)

# Group vehicles by engine size range and analyze average CO2 emissions
def engine_size_range(size):
    if size < 2.0:
        return '<2.0L'
    elif 2.0 <= size <= 3.0:
        return '2.0–3.0L'
    else:
        return '>3.0L'

co2['Engine_Size_Range'] = co2['Engine Size(L)'].apply(engine_size_range)
engine_size_emissions = co2.groupby('Engine_Size_Range')['CO2 Emissions(g/km)'].mean()
print("Average CO2 emissions by engine size range:")
print(engine_size_emissions)

Average CO2 emissions by transmission type:
Transmission_Type
Automatic    255.429240
Manual       235.889678
Name: CO2 Emissions(g/km), dtype: float64
Average CO2 emissions by engine size range:
Engine_Size_Range
2.0–3.0L    226.181397
<2.0L       180.922457
>3.0L       297.525032
Name: CO2 Emissions(g/km), dtype: float64


### Bar Chart

In [34]:
fig = px.bar(
    transmission_avg_emissions.reset_index(),
    x='Transmission_Type',
    y='CO2 Emissions(g/km)',
    color='Transmission_Type',
    title='CO2 Emissions: Manual vs Automatic Transmissions',
    labels={'CO2 Emissions(g/km)': 'Average CO2 Emissions (g/km)', 'Transmission_Type': 'Transmission Type'}
)
fig.show()

In [37]:
fig = px.box(
    co2,
    x='Engine_Size_Range',
    y='CO2 Emissions(g/km)',
    color='Engine_Size_Range',
    title='CO2 Emissions Distribution by Engine Size Range',
    labels={'Engine_Size_Range': 'Engine Size Range', 'CO2 Emissions(g/km)': 'CO2 Emissions (g/km)'}
)
fig.show()

## 3) Efficiency score: efficiency = (Combined Fuel Consumption / CO2 Emissions) and rank vehicles by it and top 10 most fuel-efficient vehicles.

In [16]:
# Create a custom efficiency score
co2['Efficiency_Score'] = co2['Fuel Consumption Comb (L/100 km)'] / co2['CO2 Emissions(g/km)']

# Top 10 most fuel-efficient vehicles
most_efficient_vehicles = co2[['Make', 'Model', 'Efficiency_Score']].sort_values(
    by='Efficiency_Score', ascending=True).head(10)
print("Top 10 most fuel-efficient vehicles:")
print(most_efficient_vehicles)

Top 10 most fuel-efficient vehicles:
            Make                      Model  Efficiency_Score
3139     PORSCHE  CAYENNE DIESEL (modified)          0.036296
3286  VOLKSWAGEN     TOUAREG TDI (modified)          0.036296
2156  VOLKSWAGEN        GOLF TDI (modified)          0.036517
2236        AUDI  A7 QUATTRO TDI (modified)          0.036620
2234        AUDI  A6 QUATTRO TDI (modified)          0.036620
1093        AUDI          A3 TDI (modified)          0.036723
2155  VOLKSWAGEN        GOLF TDI (modified)          0.036723
2175  VOLKSWAGEN      PASSAT TDI (modified)          0.036813
2020     PORSCHE  CAYENNE DIESEL (modified)          0.036823
2181  VOLKSWAGEN     TOUAREG TDI (modified)          0.036823


### Bar Chart

In [38]:
fig = px.bar(
    most_efficient_vehicles,
    x='Efficiency_Score',
    y='Model',
    orientation='h',
    color='Make',
    title='Top 10 Most Fuel-Efficient Vehicles',
    labels={'Efficiency_Score': 'Efficiency Score', 'Model': 'Vehicle Model'}
)
fig.show()

In [47]:
# Calculate average Efficiency Score by Fuel Type
fuel_efficiency_avg = co2.groupby('Fuel Type')['Efficiency_Score'].mean().reset_index()

import plotly.graph_objects as go

# Create the figure
fig = go.Figure()

# Add line chart with markers
fig.add_trace(go.Scatter(
    x=fuel_efficiency_avg['Fuel Type'],
    y=fuel_efficiency_avg['Efficiency_Score'],
    mode='lines+markers',
    name='Efficiency Trend',
    line=dict(color='royalblue', width=3),
    marker=dict(size=8)
))

# Add scatter plot for additional emphasis
fig.add_trace(go.Scatter(
    x=fuel_efficiency_avg['Fuel Type'],
    y=fuel_efficiency_avg['Efficiency_Score'],
    mode='markers',
    name='Efficiency Points',
    marker=dict(size=10, color='orange', symbol='circle')
))

# Customize the layout
fig.update_layout(
    title='Average Fuel Efficiency Score by Fuel Type',
    xaxis_title='Fuel Type',
    yaxis_title='Average Efficiency Score',
    title_x=0.5,  # Center the title
    legend=dict(title='Legend', orientation='h', x=0.5, xanchor='center', y=-0.2),
    plot_bgcolor='rgba(245, 245, 245, 1)',  # Light background for contrast
)

fig.show()

In [52]:
fig = px.scatter(
    co2,
    x='Engine Size(L)',  # Corrected column name
    y='CO2 Emissions(g/km)',
    title='Engine Size vs CO2 Emissions',
    labels={'Engine Size(L)': 'Engine Size (L)', 'CO2 Emissions(g/km)': 'CO2 Emissions (g/km)'},
    color='Fuel Type',
    size='CO2 Emissions(g/km)',
    hover_data=['Make', 'Model']
)
fig.show()

In [53]:
fig = px.scatter(
    co2,
    x='Fuel Consumption Comb (L/100 km)',
    y='CO2 Emissions(g/km)',
    color='Fuel Type',
    title='Fuel Consumption vs CO2 Emissions by Fuel Type',
    labels={'Fuel Consumption Comb (L/100 km)': 'Fuel Consumption (L/100 km)', 'CO2 Emissions(g/km)': 'CO2 Emissions (g/km)'},
    hover_data=['Make', 'Model']
)
fig.show()

In [54]:
avg_co2_by_class = co2.groupby('Vehicle Class')['CO2 Emissions(g/km)'].mean().reset_index()
fig = px.bar(
    avg_co2_by_class,
    x='Vehicle Class',
    y='CO2 Emissions(g/km)',
    title='Average CO2 Emissions by Vehicle Class',
    labels={'Vehicle Class': 'Vehicle Class', 'CO2 Emissions(g/km)': 'Average CO2 Emissions (g/km)'},
    color='Vehicle Class',
    text_auto=True
)
fig.show()